## Beyond the Score: Benchmarking Virtual Screening Integration Strategies for Balancing Recall and Precision in Early Drug Discovery

Elisabetta Grazia Tomarchio, Rocco Buccheri, Antonio Rescifina

ML-QSAR workflow


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import StratifiedShuffleSplit
 
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.MolStandardize import rdMolStandardize
 
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, auc,
    confusion_matrix, matthews_corrcoef, brier_score_loss
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils import resample
 
from tqdm import tqdm

In [ ]:
RANDOM_SEED      = 42
ECFP_BITS        = 2048
ECFP_RADIUS      = 2
ACTIVE_THRESHOLD = 6.0   # pChEMBL > 6 → attivo
N_SPLITS         = 5
 
INPUT_CSV  = "dataset.csv" # your csv
OUTPUT_DIR = "qsar_pipeline_results" # your path
MODEL_DIR  = f"{OUTPUT_DIR}/model"
FRAG_DIR   = f"{OUTPUT_DIR}/fragments"
 
for d in [OUTPUT_DIR, MODEL_DIR, FRAG_DIR]:
    os.makedirs(d, exist_ok=True)
 

In [ ]:
preview = pd.read_csv(INPUT_CSV, nrows=3)
print(preview)

## Input CSV

The input CSV file must contain at least the following columns:

- `smiles`: SMILES string of each compound.
- `pchembl_value`: Experimental pChEMBL value for each compound.

Additional columns are allowed and will be ignored unless explicitly used in the workflow.

In [ ]:
# ============================================================
# BLOCK 1 — RAW LOAD + MINIMAL CLEAN + SCAFFOLD STRATIFIED SAMPLING
# ============================================================

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from tqdm import tqdm
import os
from collections import defaultdict

tqdm.pandas()
# ---------------- USER PARAMS ----------------


N_ACTIVES_FOR_LUDE = 50
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

# ============================================================
# 1) LOAD + MINIMAL CLEAN 
# ============================================================

raw = pd.read_csv(INPUT_CSV, sep=';')
raw.columns = [c.strip().lower().replace(" ", "_") for c in raw.columns]

assert "smiles" in raw.columns
assert "pchembl_value" in raw.columns

raw = raw.dropna(subset=["smiles", "pchembl_value"]).copy()

def simple_canonical(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

print("\nCanonicalization SMILES…")

clean_rows = []
for _, row in tqdm(raw.iterrows(), total=len(raw)):
    smi = simple_canonical(row["smiles"])
    if smi is None:
        continue
    clean_rows.append({
        "smiles": smi,
        "pchembl": float(row["pchembl_value"])
    })

df = pd.DataFrame(clean_rows)

# duplicate removal
df = df.groupby("smiles", as_index=False)["pchembl"].median()

print("Dataset after clean:", len(df))

# ============================================================
# 2) LABELS
# ============================================================

df["label"] = (df["pchembl"] > ACTIVE_THRESHOLD).astype(np.int8)

actives = df[df.label == 1].copy()
inactives_real = df[df.label == 0].copy()

print("Actives total:", len(actives))
print("Inactives real:", len(inactives_real))

# ============================================================
# 3) MURCKO SCAFFOLD GENERATION
# ============================================================

def murcko(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaf)

print("\nActives scaffolds calculation…")

actives["scaffold"] = actives["smiles"].progress_apply(murcko)
actives = actives.dropna(subset=["scaffold"])

print("Unique scaffolds:", actives["scaffold"].nunique())

# ============================================================
# 4) BIN ACTIVE COMPOUNDS BY POTENCY
# ============================================================

bins = [ACTIVE_THRESHOLD, 6.5, 7, 7.5, 8, 9, 20]
labels = list(range(len(bins)-1))

actives["activity_bin"] = pd.cut(
    actives["pchembl"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# ============================================================
# 5) GROUP COMPOUNDS BY MURCKO SCAFFOLD
# ============================================================

scaffold_groups = defaultdict(list)
for idx, row in actives.iterrows():
    scaffold_groups[row["scaffold"]].append(idx)

scaffolds = list(scaffold_groups.keys())
np.random.shuffle(scaffolds)

# ============================================================
# 6) SELECT A DIVERSE SET OF ACTIVES USING
#    SCAFFOLD- AND POTENCY-STRATIFIED SAMPLING
# ============================================================


selected_indices = []

# 1. Determine the target number of compounds for each activity bin
bin_counts = actives["activity_bin"].value_counts(normalize=True)
bin_targets = {b: int(round(frac * N_ACTIVES_FOR_LUDE)) for b, frac in bin_counts.items()}

# Correct rounding errors to ensure the requested number of compounds
diff = N_ACTIVES_FOR_LUDE - sum(bin_targets.values())
if diff != 0:
    first_bin = list(bin_targets.keys())[0]
    bin_targets[first_bin] += diff

print("\n🎯 Target compounds per activity bin:", dict(bin_targets))

# 2. Build a pool of compounds grouped by activity bin and scaffold
# Structure:
# {activity_bin: {scaffold_smiles: [compound_indices]}}
pool = defaultdict(lambda: defaultdict(list))
for idx, row in actives.iterrows():
    pool[row["activity_bin"]][row["scaffold"]].append(idx)

# 3. Perform scaffold- and potency-stratified sampling
used_scaffolds_global = set()

for b_id, target in bin_targets.items():
    if target <= 0: continue
    
    bin_scaffolds = pool[b_id]
    # Sort scaffolds by the highest pChEMBL value within each activity bin
    scafs_in_bin = sorted(bin_scaffolds.keys(), 
                          key=lambda s: actives.loc[bin_scaffolds[s], "pchembl"].max(), 
                          reverse=True)
    
    selected_in_bin = []
    
    # First pass: select the highest-potency compound from each scaffold to maximize scaffold diversity.
    for s in scafs_in_bin:
        if len(selected_in_bin) >= target:
            break
        
        # Select the best molecule for each scaffold
        best_idx = actives.loc[bin_scaffolds[s], "pchembl"].idxmax()
        selected_in_bin.append(best_idx)
        used_scaffolds_global.add(s)

    # Second pass: If there are not enough unique scaffolds in the bin, randomly select additional analogs from the remaining compounds.
    if len(selected_in_bin) < target:
        remaining_in_bin = list(set([i for s in scafs_in_bin for i in bin_scaffolds[s]]) - set(selected_in_bin))
        if remaining_in_bin:
            needed = target - len(selected_in_bin)
            extra = np.random.choice(remaining_in_bin, min(len(remaining_in_bin), needed), replace=False)
            selected_in_bin.extend(extra)
            
    selected_indices.extend(selected_in_bin)

# 4. Final selection summary
actives_for_lude = actives.loc[selected_indices].copy()

print("\n--- Final Selection Summary ---")
print(f"Total compounds: {len(actives_for_lude)}")
print(f"Unique scaffolds: {actives_for_lude['scaffold'].nunique()}")
print("\nActivity bin distribution:")
print(actives_for_lude["activity_bin"].value_counts().sort_index())

# ============================================================
# 7) FINAL SPLIT
# ============================================================

actives_for_lude = actives.loc[selected_indices].copy()
actives_remaining = actives.drop(selected_indices).copy()

print("\nActives → LUDe:", len(actives_for_lude))
print("Remaining actives:", len(actives_remaining))



# ============================================================
# 8) EXPORT OUTPUT FILES 
# ============================================================

lude_input_dir = (f"{OUTPUT_DIR}/LUDe_input")
intermediate_dir = (f"{OUTPUT_DIR}/intermediate_datasets")

# Crea le cartelle
os.makedirs(lude_input_dir, exist_ok=True)
os.makedirs(intermediate_dir, exist_ok=True)

print(f"Output directories created in: {OUTPUT_DIR}")



# 1. Save the actives compounds for LUDe

lude_file_path = os.path.join(lude_input_dir, "actives_for_lude.txt")
actives_for_lude["smiles"].to_csv(lude_file_path, index=False, header=False)

# 2. Save the remaining actives compounds
remaining_actives_path = os.path.join(intermediate_dir, "actives_remaining.csv")
actives_remaining.to_csv(remaining_actives_path, index=False)

# 3. Save real inactives compounds
inactives_path = os.path.join(intermediate_dir, "inactives_real.csv")
inactives_real.to_csv(inactives_path, index=False)

# 4. Save a complete log of the copounds selected for LUDe

lude_log_path = os.path.join(intermediate_dir, "actives_for_lude_full_info.csv")
actives_for_lude.to_csv(lude_log_path, index=False)

print("\nFiles successfully saved.")
print(f"LUDe input: {lude_file_path}")
print(f"Intermediate datasets: {intermediate_dir}")

## Decoy generation

In [ ]:
# -*- coding: utf-8 -*-


"""
Input:  - a .TXT file with molecules SMILES notation, one by line.
        - a path to the folder with the ChEMBl databases
        - a path to the folder where the .TXT with SMILES is located.
        
Ouput:  - "Generated_decoys.csv" with the SMILES of all decoys  
        - "Decoys_analysis" which summarizes how many molecules passed each successive filter in the workflow
        - "Decoys_setting.csv", which contains all the settings used in the run, and the validation metrics.
"""


##### LUDe #####

# Needed packages
from pathlib import Path
import pandas as pd
import os
import sys
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, Descriptors, rdFMCS, Lipinski
from rdkit.Chem.Scaffolds import MurckoScaffold 
from molvs import Standardizer
from sklearn.metrics import  roc_curve
import numpy as np
import random
from openbabel import openbabel
import time
start = time.time()

def read_paired_file(filename):
    '''Reads .smi file '''
    '''Returns array containing smiles strings of molecules'''
    smiles, names = [], []

    with open(filename, 'r') as f:
        for line in f:
            if line:
                smiles.append(line.strip().split(' ')[0:2])
    smiles_final = pd.Series([d[0] for d in smiles])
    return smiles_final

# =============================================================================
# Queries and Folder Paths
# =============================================================================

# Name of the dataset with the queries
smiles_dataset = f"{OUTPUT_DIR}/LUDe_input/actives_for_lude.txt"
# Folder containing the dataset with the queries
directory = OUTPUT_DIR
loaded_smiles = read_paired_file(smiles_dataset)

# Folder with ChEMBL dataset
directory_chembl = str(Path(f"/Users/elisabettatomarchio/Desktop/nuovo_progetto/training/LUDe.v1.0/Base_ChEMBL_final"))

# =============================================================================
# Customizable Options
# =============================================================================

# Physicochemical features limits
lim_MW = 20         # Molecular Weight
lim_logP = 0.5      # logP
lim_rb = 1   # Rotable bonds
lim_Hba = 1    # Num of H Acceptors
lim_Hbd = 1      # Num of H Donors
lim_charge = 1      # Formal charge

# Dissimilarty conditions
fingerprint_radio =  2          # 1, 2, 3
fingerprint_lenght =  1024      # 512, 1024, 2048
similarity_metric = "TanimotoSimilarity" # "TanimotoSimilarity", "DiceSimilarity", "CosineSimilarity", "SokalSimilarity", "RusselSimilarity", "KulczynskiSimilarity", "McConnaugheySimilarity"
max_similarity_limit = 0.2      # Maximum allowed similarity value: from 0.0 to 1.0

# Limit of the fraction of the Maximum Common Substructure
lim_fraction_MCS = 0.5 # from 0.0 to 1.0

# Decoys with different framework. Options "True" or "False"
framework_filter = True

# Maximum similarity allowed between decoys and any of the actives
max_similarity_limit_all = 0.2 # from 0.0 to 0.7

# Max number of decoys by active compound
max_decoys = 50 

# =============================================================================
# LUDe script
# =============================================================================

#%%
def charges_ph(molecule):
    
    # obConversion it's neccesary for saving the objects
    obConversion = openbabel.OBConversion()
    obConversion.SetInAndOutFormats("smi", "smi")
    
    # create the OBMol object and read the SMILE
    mol = openbabel.OBMol()
    obConversion.ReadString(mol, molecule)
    
    # Add H, correct pH and add H again, it's the only way it works
    mol.AddHydrogens()
    mol.CorrectForPH(7.4)
    mol.AddHydrogens()
    
    # transforms the OBMOl objecto to string (SMILES)
    optimized = obConversion.WriteString(mol)
    
    return optimized

#%%

def smile_obabel_corrector(smiles_ionized):
    mol1 = Chem.MolFromSmiles(smiles_ionized, sanitize = False)
    
    # checks if the ether group is wrongly protonated
    pattern1 = Chem.MolFromSmarts('[#6]-[#8-]-[#6]')
    if mol1.HasSubstructMatch(pattern1):
        # gets the atom number for the O wrongly charged
        at_matches = mol1.GetSubstructMatches(pattern1)
        at_matches_list = [y[1] for y in at_matches]
        # changes the charged for each O atom
        for at_idx in at_matches_list:
            atom = mol1.GetAtomWithIdx(at_idx)
            atom.SetFormalCharge(0)
            atom.UpdatePropertyCache()

    pattern12 = Chem.MolFromSmarts('[#6]-[#8-]-[#16]')
    if mol1.HasSubstructMatch(pattern12):
        # gets the atom number for the O wrongly charged
        at_matches = mol1.GetSubstructMatches(pattern12)
        at_matches_list = [y[1] for y in at_matches]
        # changes the charged for each O atom
        for at_idx in at_matches_list:
            atom = mol1.GetAtomWithIdx(at_idx)
            atom.SetFormalCharge(0)
            atom.UpdatePropertyCache()
            
    # checks if the nitro group is wrongly protonated in the oxygen
    pattern2 = Chem.MolFromSmarts('[#6][O-]=[N+](=O)[O-]')
    if mol1.HasSubstructMatch(pattern2):
        # print('NO 20')
        patt = Chem.MolFromSmiles('[O-]=[N+](=O)[O-]', sanitize = False)
        repl = Chem.MolFromSmiles('O[N+]([O-])=O')
        rms = AllChem.ReplaceSubstructs(mol1,patt,repl,replaceAll=True)
        mol1 = rms[0]

    # checks if the nitro group is wrongly protonated in the oxygen
    pattern21 = Chem.MolFromSmarts('[#6]-[O-][N+](=O)=[O-]')
    if mol1.HasSubstructMatch(pattern21):
        # print('NO 21')
        patt = Chem.MolFromSmiles('[O-][N+](=O)=[O-]', sanitize = False)
        repl = Chem.MolFromSmiles('[O][N+](=O)-[O-]')
        rms = AllChem.ReplaceSubstructs(mol1,patt,repl,replaceAll=True)
        mol1 = rms[0]
        
    # checks if the nitro group is wrongly protonated, different disposition of atoms
    pattern22 = Chem.MolFromSmarts('[#8-][N+](=[#6])=[O-]')
    if mol1.HasSubstructMatch(pattern22):
        # print('NO 22')
        patt = Chem.MolFromSmiles('[N+]([O-])=[O-]', sanitize = False)
        repl = Chem.MolFromSmiles('[N+]([O-])-[O-]')
        rms = AllChem.ReplaceSubstructs(mol1,patt,repl,replaceAll=True)
        mol1 = rms[0]

    # checks if the nitro group is wrongly protonated, different disposition of atoms
    pattern23 = Chem.MolFromSmarts('[#6][N+]([#6])([#8-])=[O-]')
    if mol1.HasSubstructMatch(pattern23):
        # print('NO 23')
        patt = Chem.MolFromSmiles('[N+]([O-])=[O-]', sanitize = False)
        repl = Chem.MolFromSmiles('[N+]([O-])[O-]')
        rms = AllChem.ReplaceSubstructs(mol1,patt,repl,replaceAll=True)
        mol1 = rms[0]

    # checks if the nitro group is wrongly protonated, different disposition of atoms
    pattern24 = Chem.MolFromSmarts('[#6]-[#8][N+](=O)=[O-]')
    if mol1.HasSubstructMatch(pattern24):
        # print('NO 24')
        patt = Chem.MolFromSmiles('[O][N+](=O)=[O-]', sanitize = False)
        repl = Chem.MolFromSmiles('[O][N+](=O)[O-]')
        rms = AllChem.ReplaceSubstructs(mol1,patt,repl,replaceAll=True)
        mol1 = rms[0]

    # checks if the 1H-tetrazole group is wrongly protonated
    pattern3 = Chem.MolFromSmarts('[#7]-1-[#6]=[#7-]-[#7]=[#7]-1')
    if mol1.HasSubstructMatch(pattern3):
        # gets the atom number for the N wrongly charged
        at_matches = mol1.GetSubstructMatches(pattern3)
        at_matches_list = [y[2] for y in at_matches]
        # changes the charged for each N atom
        for at_idx in at_matches_list:
            atom = mol1.GetAtomWithIdx(at_idx)
            atom.SetFormalCharge(0)
            atom.UpdatePropertyCache()

    # checks if the 2H-tetrazole group is wrongly protonated
    pattern4 = Chem.MolFromSmarts('[#7]-1-[#7]=[#6]-[#7-]=[#7]-1')
    if mol1.HasSubstructMatch(pattern4):
        # gets the atom number for the N wrongly charged
        at_matches = mol1.GetSubstructMatches(pattern4)
        at_matches_list = [y[3] for y in at_matches]
        # changes the charged for each N atom
        for at_idx in at_matches_list:
            atom = mol1.GetAtomWithIdx(at_idx)
            atom.SetFormalCharge(0)
            atom.UpdatePropertyCache()
        
    # checks if the 2H-tetrazole group is wrongly protonated, different disposition of atoms
    pattern5 = Chem.MolFromSmarts('[#7]-1-[#7]=[#7]-[#6]=[#7-]-1')
    if mol1.HasSubstructMatch(pattern5):
        # gets the atom number for the N wrongly charged
        at_matches = mol1.GetSubstructMatches(pattern4)
        at_matches_list = [y[4] for y in at_matches]
        # changes the charged for each N atom
        for at_idx in at_matches_list:
            atom = mol1.GetAtomWithIdx(at_idx)
            atom.SetFormalCharge(0)
            atom.UpdatePropertyCache()
    smile_checked = Chem.MolToSmiles(mol1)
   
    return smile_checked

#%%
def Standardization(molecula_ok, i, s):

    try:
        mol = Chem.MolFromSmiles(molecula_ok) # convierte las moleculas a mol
        mol_standarizado = s.fragment_parent(mol) #Return the fragment parent of a given molecule, the largest organic covalent unit in the molecule
        # mol_standarizado = s.stereo_parent(mol_standarizado, skip_standardize= True) #Return The stereo parentof a given molecule, has all stereochemistry information removed from tetrahedral centers and double bonds.
        mol_standarizado = s.charge_parent(mol_standarizado, skip_standardize= True) #Return the charge parent of a given molecule,  the uncharged version of the fragment parent
        mol_standarizado = s.isotope_parent(mol_standarizado, skip_standardize= True) #Return the isotope parent of a given molecule, has all atoms replaced with the most abundant isotope for that element.
       
        smile_standarizado = Chem.MolToSmiles(mol_standarizado)
        # ionized_smile = Chem.MolToSmiles(mol_standarizado)

        ionized_smile = charges_ph(smile_standarizado)
        smile_checked = smile_obabel_corrector(ionized_smile)
        
        mol_checked = Chem.MolFromSmiles(smile_checked)
    except:
        print("**Oh no! There is a problem with standarization of one SMILES.**")
        print("**Please check your molecule: **" + str(i))
        print("**That is the SMILES: **" + str(molecula_ok))
        sys.exit()
    
    return mol_checked

#%%

def cal_descriptors_DB(mol_standarizado):

    # Calculate properties and store in dict
    prop_dict = {}
    # molweight
    prop_dict.update({'MW': Descriptors.MolWt(mol_standarizado)})
    # logP
    prop_dict.update({'LogP': Chem.Crippen.MolLogP(mol_standarizado)})
    # HBA
    prop_dict.update({'Hba': Chem.rdMolDescriptors.CalcNumLipinskiHBA(mol_standarizado)})
    # HBD
    prop_dict.update({'Hbd': Chem.rdMolDescriptors.CalcNumLipinskiHBD(mol_standarizado)})
    # rotatable bonds
    prop_dict.update({'rb': Chem.rdMolDescriptors.CalcNumRotatableBonds(mol_standarizado)})
    # Formal (net) charge
    prop_dict.update({'charge': Chem.rdmolops.GetFormalCharge(mol_standarizado)})

    size_molec = Descriptors.HeavyAtomCount(mol_standarizado)
    
    # molec_descrip = pd.Series({'MW' : MolWt, 'LogP': MolLogP, 'Num_Rotatable_Bonds': NumRotatableBonds,'Num_H_Acceptors': NumHAcceptors, 'Num_H_Donors': NumHDonors})

    return prop_dict, size_molec


#%%
def Physicochem_distances(df_only_property, molec_descrip):
    '''
    Calculates the maximum distances for the five Physicochem descriptors
    between the query molecule and all the selected decoy from the Physicochem df
    '''
    distances = []
    
    physicochem_min = pd.Series([df_only_property[column].min() for column in df_only_property],
                                index = ['MW', 'LogP', 'rb', 'Hba', 'Hbd', 'charge'])
    physicochem_max = pd.Series([df_only_property[column].max() for column in df_only_property],  
                                index = ['MW', 'LogP', 'rb', 'Hba', 'Hbd', 'charge'])
    for idx, value in physicochem_min.items():
        if abs(molec_descrip[idx] - physicochem_min[idx]) < abs(molec_descrip[idx] - physicochem_max[idx]):
            distances.append(abs(molec_descrip[idx] - physicochem_max[idx]))
        else:
            distances.append(abs(molec_descrip[idx] - physicochem_min[idx]))
    
    distances_serie = pd.Series(distances)
    distances_serie.rename(index = {0:'f_MW' , 1:'f_LogP', 2:'f_rb',3:'f_Hba', 4:'f_Hbd', 5:'f_charge'}, inplace=True)
    
    return distances_serie

#%%

def Calculate_PSS(df_decoys_physicochem, molec_descrip):
    ''' 
    Calculates the physicochemical similarity score (PSS)
    for each decoy according to their query molecule
    '''
    # for each Active row with their physicochem properties
    df_final = pd.DataFrame()
    relative_distance= []
    
    df_only_property = df_decoys_physicochem.copy()    
    df_only_property.drop(labels = ['SMILE', 'Framework'], axis = 1, inplace = True)
    
    df_only_framework = df_decoys_physicochem.copy()    
    df_only_framework.drop(labels = ['MW', 'LogP', 'rb', 'Hba', 'Hbd', 'charge'], axis = 1, inplace = True)
    
    # Calculates the max distance between the query_i and all the decoys for the Physicochem prop
    distances = Physicochem_distances(df_only_property, molec_descrip)
    
    # Iterate for each property and distance of the active
    for prop, value in molec_descrip.items():
        relative_distance_prop = []
        f_prop = distances[f'f_{prop}']
        decoy_prop_physicochem = df_only_property[prop]
        for x in decoy_prop_physicochem:
            if f_prop == 0:
                relative_distance_prop.append(1)
            else:
                normalization = 1 - (abs(value - x) / f_prop)
                relative_distance_prop.append(normalization)
        relative_distance.append(relative_distance_prop)
        
    df_dist = pd.DataFrame(relative_distance)
    df_final = pd.concat([df_final, df_dist], axis = 1, ignore_index= True)
   
    df_final = df_final.transpose()
    df_final = df_final.set_index(df_only_framework.index)
    df_only_framework['PSS'] = df_final.mean(axis = 1)
    
    return df_only_framework

#%%

def Dissimilarity_filter(df_decoys_PSS, fp_1, mol_standarizado, framework_query, size_molec, smiles_seleccionados, decoy_by_active, filtro_fw, filtro_MCS, filtro_tanimoto, nombre, database_random):
    ''' 
    Dissimilarity conditions for each decoy with molecule "Query_i", which involves:
    filter by Maximum allowed similarity value, calculating Tanimoto similarity
    filter by limit of the fraction of the Maximum Common Substructure
    filter by framework if the framework of the decoys are the same as the Query
    '''
    print(f'Calculating Maximum allowed similarity and MCS for {database_random}')
    
    # filter by Maximum allowed similarity value, calculating Tanimoto similarity
            
    for index, decoy_in_df in df_decoys_PSS.iterrows():
    
        smiles_DB = decoy_in_df['SMILE']
        fp_2 = AllChem.GetMorganFingerprintAsBitVect(decoy_in_df["mol"],fingerprint_radio, 
                                                     nBits = fingerprint_lenght, useFeatures=False)
        similarity_metric_ok = getattr(DataStructs, similarity_metric)
        tan_sim = similarity_metric_ok(fp_1, fp_2)
        
        # Tanimoto similarity Maximum allowed similarity value
        if tan_sim <= float(max_similarity_limit):
            mols = [mol_standarizado,decoy_in_df["mol"]]
            filtro_tanimoto.append(smiles_DB)
            res = rdFMCS.FindMCS(mols, timeout=10)
            tamanio_MCS = res.numAtoms
            
            # Limit of the fraction of the Maximum Common Substructure
            if tamanio_MCS / size_molec < lim_fraction_MCS:
                filtro_MCS.append(smiles_DB)
                if framework_filter == True:
                    framework_DB = decoy_in_df['Framework']
                    if framework_query != framework_DB:
                        if not smiles_DB in smiles_seleccionados:
                            decoy_by_active.append(pd.Series({'SMILE': smiles_DB, 'fp':fp_2, 'Query': nombre}))
                        filtro_fw.append(smiles_DB)
                        smiles_seleccionados.append(smiles_DB)
                    else:
                        pass
                else:
                    if not smiles_DB in smiles_seleccionados:
                        decoy_by_active.append(pd.Series({'SMILE': smiles_DB, 'fp':fp_2, 'Query': nombre}))
                    filtro_fw.append(smiles_DB)
                    smiles_seleccionados.append(smiles_DB)
                 
    
     
    return filtro_fw, filtro_MCS, filtro_tanimoto


#%%

def decoy_fase1(loaded_smiles, verbose = False):
        
    # my_molecules = loaded_smiles[0].tolist()
    my_molecules = loaded_smiles.tolist()
    active_standarized_smiles = []
    smiles_seleccionados = []
    fp_query = []
    df_analysis = pd.DataFrame()   
    s = Standardizer()
    df_decoy_complete = pd.DataFrame()
    
    for i, molecules in enumerate(my_molecules,start = 1):
       
        # Update progress bar
        if verbose:
            print(f"\rProgress: {str(i)} / {str(len(my_molecules))}" )
        
        nombre = "Query_" + str(i)
        
        filtro_physicochemical = [] # molecules passing physicochemical filters
        filtro_tanimoto=[]          # molecules passing similarity filter        
        filtro_MCS=[]               # molecules passing MCS filter
        filtro_fw=[]                # molecules passing framework filter
        decoy_by_active = []

        molecula_ok = molecules.strip()
        
        # Standardization
        mol_standarizado = Standardization(molecula_ok, i, s)
        active_standarized_smiles.append(Chem.MolToSmiles(mol_standarizado))
        # Framework
        core = MurckoScaffold.GetScaffoldForMol(mol_standarizado)
        framework_query = Chem.MolToSmiles(core)
    
        # DESCRIPTORES
        molec_descrip, size_molec = cal_descriptors_DB(mol_standarizado)
        
        # FINGERPRINT
        fp_1 = AllChem.GetMorganFingerprintAsBitVect(mol_standarizado,fingerprint_radio,nBits = fingerprint_lenght,useFeatures=False)
        fp_query.append(fp_1)
        
        # BASE DE DATOS
        # random seed a partir del i de Query, por eso cambia la primera database para cada Query
        # siempre la primer Query va a tener seed = 1, por lo que seleciona df_decoys_physicochem mismo orden
        random.seed(i) 
        total_databases = os.listdir(directory_chembl)
        while total_databases:  
            round_physicochemical = 1
            df_decoys_physicochem = pd.DataFrame()
            # tomo una base de datos random del total, asi no empiezo siempre con la primera
            index = random.randrange(len(total_databases))
            database_random = total_databases[index]
            del total_databases[index]
            df_database = pd.read_csv(directory_chembl + '/' + database_random, sep="\t", index_col=False, header='infer')
            
            # instead of workig with the descriptors value, we worked with the difference 
            # between the decoys molecules and the Query molecule 
            df_database['MW'] = abs(df_database['MW'] - molec_descrip['MW'])
            df_database['LogP'] = abs(df_database['LogP'] - molec_descrip['LogP'])
            df_database['rb'] = abs(df_database['rb'] - molec_descrip['rb'])
            df_database['Hba'] = abs(df_database['Hba'] - molec_descrip['Hba'])
            df_database['Hbd'] = abs(df_database['Hbd'] - molec_descrip['Hbd'])
            df_database['charge'] = abs(df_database['charge'] - molec_descrip['charge'])
            
            lim_MW_DB, lim_logP_DB, lim_rb_DB, lim_Hba_DB, lim_Hbd_DB, lim_charge_DB = lim_MW, lim_logP, lim_rb, lim_Hba, lim_Hbd, lim_charge
            
            while len(df_decoys_physicochem) < 400 or round_physicochemical < 5:
                round_physicochemical = round_physicochemical + 1
                # Applying Physicochemical features limits
                df_decoys_physicochem = df_database[
                    (df_database['MW'] <= lim_MW_DB) &
                    (df_database['LogP'] <= lim_logP_DB) &
                    (df_database['rb'] <= lim_rb_DB) &
                    (df_database['Hba']<= lim_Hba_DB) &
                    (df_database['Hbd']<= lim_Hbd_DB) &
                    (df_database['charge']<= lim_charge_DB)].copy()

                # if there are more than 400 molecules than pass the Physicochemical filter 
                # if there are less than 400 molecules the Physicochemical limits are widen
                if len(df_decoys_physicochem) < 400:
                    # print('The limites were extended')
                    lim_MW_DB =  lim_MW * (round_physicochemical + 1) / 2
                    lim_logP_DB =  lim_logP * (round_physicochemical + 1) / 2
                    lim_rb_DB =  lim_rb * (round_physicochemical + 1) / 2
                    lim_Hba_DB =  lim_Hba * (round_physicochemical + 1) / 2
                    lim_Hbd_DB = lim_Hbd * (round_physicochemical + 1) / 2 
                    lim_charge_DB =  lim_charge * (round_physicochemical + 1) / 2
                    # print(round_physicochemical, lim_MW_DB, lim_logP_DB, lim_rb_DB, lim_Hba_DB, lim_Hbd_DB, lim_charge_DB)
                else:
                    # Calculates the PSS Score for the decoys 
                    df_decoys_PSS = Calculate_PSS(df_decoys_physicochem, molec_descrip) 
                    # Only keeps the top 200 compounds with higher PSS
                    df_decoys_PSS.sort_values(by = 'PSS', ascending = False, inplace = True)
                    df_decoys_PSS = df_decoys_PSS.head(200)
                    break

            filtro_physicochemical.append(len(df_decoys_PSS))
            
            # Calculates MolFromSmiles from selected decoys with Physicochemical
            df_decoys_PSS['mol'] = df_decoys_PSS['SMILE'].apply(lambda x: Chem.MolFromSmiles(x))  ['mol'] = df_decoys_physicochem['SMILE'].apply(lambda x: Chem.MolFromSmiles(x))  
            
            # Dissimilarity filter for each decoy with molecule "Query_i"
            filtro_fw, filtro_MCS, filtro_tanimoto = Dissimilarity_filter(df_decoys_PSS, fp_1, 
                                            mol_standarizado, framework_query, size_molec, smiles_seleccionados, 
                                            decoy_by_active, filtro_fw, filtro_MCS, filtro_tanimoto, nombre, database_random)
            
            print(database_random)
            print('Decoys per active that pass physico and similarity', len(decoy_by_active))
            # instead of having too many decoys we only keep 500 
            if len(decoy_by_active) > 500:
                break

        # after iteration of all the ChEMBL databases or if after the filster 300 decoys per active are reached
        serie_OK = pd.Series({"Query": nombre, "Selected by physicochemical properties": sum(filtro_physicochemical),
                              "Pass the Tc filter per active": len(filtro_tanimoto),"Pass the fMCS filter": len(filtro_MCS), 
                              "Pass the Framework filter": len(filtro_fw), 'Non duplicated Decoys per active:' : len(decoy_by_active)})

        df_decoy_complete = pd.concat([df_decoy_complete, pd.DataFrame(decoy_by_active)], axis = 0, ignore_index= True)
        
        df_analysis = pd.concat([df_analysis, serie_OK], axis = 1, ignore_index=True)
    
    df_analysis = df_analysis.transpose()
    df_analysis.set_index('Query', drop=True, inplace = True)
    
    print('')
    print('Decoys have been successfully obtained for each loaded molecule!!')
    print("---------------------------------------------------------------")
    print("Molecules that passed the physicochemical properties filters: " + str(df_analysis['Selected by physicochemical properties'].sum()))
    print("Molecules that passed the dissimilarity (structural) filters: " + str(len(smiles_seleccionados)))
    print("Of which: " + str(df_decoy_complete.shape[0]) + " are different")

    return (df_decoy_complete, fp_query, df_analysis, active_standarized_smiles)


#%% To take XXX random decoys from each list

def duplicates_filter(df_decoy_complete, fp_query, df_analysis):
    final_decoys = pd.DataFrame()

    # Comparing actives and decoys by tanimoto
    df_Tanimoto_final = Tanimoto_actives_decoys(df_decoy_complete, fp_query, max_similarity_limit_all)
    number_decoys_TC = df_Tanimoto_final.value_counts(subset = ['Query'], sort = False)
    number_decoys_TC.rename('Pass the TC filter total actives:', inplace = True)
    df_analysis_final = pd.merge(df_analysis, number_decoys_TC,how = 'left', left_on = 'Query',right_on = 'Query')
    
    # keeping only max_decoys per Query_j, random selection
    for j, fp_ in enumerate(fp_query, start = 1):
        decoys_from_query = df_Tanimoto_final[df_Tanimoto_final['Query'] == f'Query_{j}'] 
        if decoys_from_query.shape[0] >= max_decoys:
            df_decoys_max = decoys_from_query.sample(n = max_decoys)
        else:
            df_decoys_max = decoys_from_query

        final_decoys = pd.concat([final_decoys, df_decoys_max], axis = 0, ignore_index= True)
        
    print("Finally, " + str(df_Tanimoto_final.shape[0]) + " passed the Tanimoto filter by comparing all loaded molecules")
    print('')
    print('Congratulations, you have obtained ' + str(final_decoys.shape[0]) + " decoys!!!")
    print("---------------------------------------------------------------")
    final_decoys.drop(labels = ['fp'], axis = 1, inplace = True)
    return final_decoys, df_analysis_final

#%%

def Tanimoto_actives_decoys(df_decoy_complete, fp_query, max_similarity_limit_all):
    ''' 
    From the df with decoys that pass the filters,
    only the rows that are dissimilar to all the actives are kept
    '''
    Tanimoto_all_decoy = []
    
    for _, decoy_in_df in df_decoy_complete.iterrows():
        coef_tan= []
        for fp_activo in fp_query:
            tan_sim=DataStructs.TanimotoSimilarity(decoy_in_df['fp'], fp_activo)
            coef_tan.append(tan_sim)
        if max(coef_tan) < max_similarity_limit_all:
            Tanimoto_all_decoy.append(decoy_in_df)
    df_Tanimoto_final = pd.DataFrame(Tanimoto_all_decoy)    
    
    return df_Tanimoto_final


#%%

lista_resultados = decoy_fase1(loaded_smiles, verbose = True)
print("Now, we are comparing all decoys vs all input SMILES, please wait a moment...")
df_final_decoys, df_analysis = duplicates_filter(lista_resultados[0], lista_resultados[1], lista_resultados[2]) 

# Here you can dowload the generated decoys
df_final_decoys.to_csv(f'{OUTPUT_DIR}/Generated_decoys.csv',index=False,header=True)

# Here you can see a little analysis of the process
df_analysis.to_csv(f'{OUTPUT_DIR}/Decoys_analysis.csv',index=True,header=True)

# Here you can download your settings


end = time.time()
hours, rem = divmod(end-start, 3600)
minutes, seconds = divmod(rem, 60)
# print("{:0>2}:{:0>2}:{:05.2f}".format(int(hours),int(minutes),seconds)) 

#%%

def Doppelganger_score_calculation(decoys_fp, actives_fp):
    ''' 
    Following Vogel_2011 DEKOIS
    From the decoys Series with their FP compares with the actives Series with FP
    and calculates the similarity matrix, selecting the max value 
    
    En DeepCoy dice: 
    For each decoy molecule, its doppelganger score is the maximum similarity across all actives
    '''
    time_start = time.time()
    Tanimoto_all_decoy = []
    print("#"*50)
    print('METRIC CALCULATION\n')
    print('**Doppelganger Score**')
    # Creates the similarity matrix for Query and Decoys
    for fp_decoy in decoys_fp:
        coef_tan = []
        for fp_activo in actives_fp:
            tan_sim=DataStructs.TanimotoSimilarity(fp_decoy, fp_activo)
            coef_tan.append(tan_sim)
        Tanimoto_all_decoy.append(coef_tan) 
    df_Tanimoto_final = pd.DataFrame(Tanimoto_all_decoy)
    
    # Determine the maximum TC for each Query atom to all Decoys
    max_TC_decoys = df_Tanimoto_final.max(axis = 0)
    
    max_doppelganger_score = max_TC_decoys.max()
    doppelganger_score = max_TC_decoys.mean()
    
    print(f'Doppelganger Score calculation took {round(time.time()-time_start)} seconds')
    print('-'*50)
    return df_Tanimoto_final, doppelganger_score, max_doppelganger_score

df_Tanimoto_final, doppelganger_score, max_doppelganger_score = Doppelganger_score_calculation(lista_resultados[0]["fp"], lista_resultados[1])


#%%

def cal_descriptors(mol_standarizado):
    ''' 
    Calculates the same 6 molecular descriptors as in Dud-E
    las query de Alexander calculaba HBD y HBA de dos formas con valores diferentes :
        Chem.Lipinski.NumHAcceptors()
        rdMolDescriptors.CalcNumLipinskiHBA()
    Según mails con Alexander, Lipinski.NumHAcceptors da mejores resultados
    para generar los LUDe empleamos esa funcion y por lo tanto aca tmb
    '''
    
    MolWt = Descriptors.MolWt(mol_standarizado) 
    MolLogP =  Chem.Crippen.MolLogP(mol_standarizado)
    NumRotatableBonds = Lipinski.NumRotatableBonds(mol_standarizado) 
    NumHAcceptors =  Lipinski.NumHAcceptors(mol_standarizado) 
    NumHDonors =  Lipinski.NumHDonors(mol_standarizado) 
    Formalcharge = Chem.rdmolops.GetFormalCharge(mol_standarizado)
    
    molec_descrip = [MolWt, MolLogP, NumRotatableBonds, NumHAcceptors,
                     NumHDonors, Formalcharge]
    return molec_descrip



def dataset_preparation(dataset_smiles, name: str):
    ''' 
    Given the Query or the decoys dataset
    standardize the molecules, calculates the FP and the molecular descriptors
    '''
    time_start = time.time()
    print("#"*50)
    print(f'Descriptor calculation of {name}s, no Standardization\n')
    print(f'There are a total of {len(dataset_smiles)} {name} molecules ')
    
    dataset_mol = [Chem.MolFromSmiles(smile) for smile in dataset_smiles]       
    dataset_desc_np = np.array([cal_descriptors(mol_active) for mol_active in dataset_mol])
    
    print(f'\nPreparation of the {name}_dataset took {round(time.time()-time_start)} seconds\n')

    return dataset_desc_np

actives_desc = dataset_preparation(lista_resultados[3], 'Query')
decoys_desc = dataset_preparation(lista_resultados[3], 'Decoy')

def normalize_descriptors_percentile(all_desc):
    ''' 
    Following Vogel_2011 DEKOIS
    for each descriptor, calculates the 95th and the 5th percentile of all the molecules
    then do the rest between the 95th and the 5th percentiles and
    normalizes the descriptors values with the aforementioned difference
    '''
    normalized_all_desc = pd.DataFrame()
    percentile = all_desc.quantile(0.95) - all_desc.quantile(0.05)
        
    for column in all_desc:
        normalized_all_desc[column] = all_desc[column]/percentile[column]
        
    return normalized_all_desc


#%%

def doe_score(actives, decoys):
    ''' 
    Following Imrie_2021 DeepCoy
    Calculates the DOE score with the script from the paper
    for every active measure the distance with every molecule
    calculates the AUC ROC 
    '''
    time_start = time.time()
    print('**DOE Score**')
    print('Calculating the Areas Between the Curves and ROC for each Query molecule')
    all_feat = list(actives) + list(decoys)
    up_p = np.percentile(all_feat, 95, axis=0)
    low_p = np.percentile(all_feat, 5, axis=0)
    norms = up_p - low_p
    for i in range(len(norms)):
        if norms[i] == 0:
            norms[i] = 1.

    active_norm = [act/norms for act in actives]
    decoy_norm = [dec/norms for dec in decoys]
    all_norm = active_norm + decoy_norm

    active_embed = []
    labels = [1] * (len(active_norm)-1) + [0] * len(decoy_norm)
    for i, act in enumerate(active_norm):
        comp = list(all_norm)
        del comp[i]
        dists = [100 - np.linalg.norm(c-act) for c in comp] # arbitrary large number to get scores in reverse order
        fpr, tpr, _ = roc_curve(labels, dists)
        fpr = fpr[::]
        tpr = tpr[::]
        a_score = 0
        for i in range(len(fpr)-1):
            a_score += (abs(0.5*( (tpr[i+1]+tpr[i])*(fpr[i+1]-fpr[i]) - (fpr[i+1]+fpr[i])*(fpr[i+1]-fpr[i]) )))
        active_embed.append(a_score)

    #print(np.average(active_embed))
    print(f'DOE Score calculation took {round(time.time()-time_start)} seconds\n')
    return np.average(active_embed)


DOE_score_np = doe_score(actives_desc, decoys_desc)


#%% Settings
def setting_info():
    from datetime import date
    today = date.today()
    fecha = today.strftime("%d/%m/%Y")
    settings = []
    settings.append(["Decoys generated at: " , fecha])
    settings.append(["\n"])
    settings.append(["Physicochemical features limits:",""])
    settings.append(["MW" , "+/- " +  str(lim_MW)])
    settings.append(["logP" , "+/- " + str(lim_logP)])
    settings.append(["Num_rotable_bonds" , "+/- " + str(lim_rb)])
    settings.append(["Num_H_acceptors" , "+/- " + str(lim_Hba)])
    settings.append(["Num_H_donors" , "+/- " + str(lim_Hbd)])
    settings.append(["\n"])
    settings.append(["Topological features ---> Dissimilarty conditions",""])
    settings.append(["Morgan Fingerprints",""])
    settings.append(["Fingerprint radio:" , str(fingerprint_radio)])
    settings.append(["Fingerprint lenght:" , str(fingerprint_lenght)])
    if lim_fraction_MCS == 1.0:
        settings.append(["Not set a limit for the fraction of the Maximum Common Substructure",""])
    else:
        settings.append(["Limit of fraction of the Maximum Common Substructure: " , str(lim_fraction_MCS)])
    settings.append(["Decoys with different framework:" , str(framework_filter)])
    settings.append(["Max Tc similarity between decoys and other actives:", str(max_similarity_limit)])
    settings.append(["Max number of decoys by loaded molecule:", str(max_decoys)])
    settings.append(["\n"])
    settings.append(["Decoy validation", " "])
    settings.append(["DOE_score", str(DOE_score_np)])
    settings.append(["Mean Doppelganger_score", str(doppelganger_score)])
    settings.append(["Max Doppelganger_score", str(max_doppelganger_score)])
    settings_df = pd.DataFrame(settings)
    return settings_df


settings_df = setting_info()  
settings_df.to_csv(f'{OUTPUT_DIR}/Decoys_settings.csv',index=False,header=False)


#%%

print("Validation metrics:")
print("\n")

print(f"{'Metric':<25}{'Value':<10}{'Comment':<10}")
print("-" * 85)
print(f"{'DOE_score':<25}{round(DOE_score_np, 3):<10}{'best possible score = 0, worst possible score = 0.5':<10}")
print(f"{'Doppelganger score':<25}{round(doppelganger_score, 3):<10}{'best possible score = 0, worst possible score = 1':<10}")
print(f"{'Max Doppelganger score':<25}{round(max_doppelganger_score, 3):<10}{'best possible score = 0, worst possible score = 1':<10}")




## DATA CURATION E FINGERPRINT CALCULATION

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize
from tqdm import tqdm
import os
from collections import defaultdict

# ─────────────────────────────────────────────────────────────
# CONFIGURATION AND PARAMETERS
# ─────────────────────────────────────────────────────────────
tqdm.pandas()
np.random.seed(42)

# Maximum allowed discrepancy:
# if the difference between the maximum and minimum pChEMBL
# values for the same SMILES is > 1.2, the compound is discarded.
PCHEMBL_MAX_DISCREPANCY = 1.2

# ─────────────────────────────────────────────────────────────
# CURATION AND QUALITY CONTROL FUNCTIONS
# ─────────────────────────────────────────────────────────────

fragment_remover = rdMolStandardize.FragmentRemover()
uncharger = rdMolStandardize.Uncharger()
normalizer = rdMolStandardize.Normalizer()
tautomer_enumerator = rdMolStandardize.TautomerEnumerator()

def rename_smiles_column(df):
    """Search for common SMILES column names and rename them to 'smiles'."""
    possible_names = ['smiles', 'SMILES', 'SMILE', 'canonical_smiles', 'Smile']
    for name in possible_names:
        if name in df.columns:
            return df.rename(columns={name: 'smiles'})
    # If no valid SMILES column is found, print a warning
    print(f"⚠️ Warning: no SMILES column found among {df.columns.tolist()}")
    return df

def standardize_molecule(smiles):
    """RDKit standardization pipeline: salt removal, normalization, uncharging, and tautomer canonicalization."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        mol = fragment_remover.remove(mol)
        mol = normalizer.normalize(mol)
        mol = uncharger.uncharge(mol)
        Chem.SanitizeMol(mol)
        mol = tautomer_enumerator.Canonicalize(mol)
        return Chem.MolToSmiles(mol, canonical=True)
    except:
        return None

def quality_median(series):
    """Compute the median only if replicate pChEMBL values are sufficiently consistent."""
    if len(series) > 1:
        diff = series.max() - series.min()
        if diff > PCHEMBL_MAX_DISCREPANCY:
            return np.nan  # Will be removed later using dropna
    return series.median()

def process_database(df, name):
    """Apply molecular standardization, quality filtering, and column selection."""
    print(f"\n🧼 Chemical curation and quality filtering: {name}...")

    # 1. Perform molecular standardization
    df["smiles"] = df["smiles"].progress_apply(standardize_molecule)
    df = df.dropna(subset=["smiles"])

    # 2. Handle missing pChEMBL values (e.g., decoys)
    if "pchembl" not in df.columns:
        df["pchembl"] = np.nan

    # 3. Remove duplicates using quality-controlled pChEMBL aggregation
    df_clean = df.groupby("smiles", as_index=False).agg({
        "label": "max",
        "pchembl": quality_median
    })

    # Remove inconsistent active compounds (NaN returned by quality_median)
    # while keeping decoys (which have NaN pChEMBL and label = 0)
    df_clean = df_clean[~((df_clean['label'] == 1) & (df_clean['pchembl'].isna()))]

    return df_clean[["smiles", "pchembl", "label"]]

# ─────────────────────────────────────────────────────────────
# LOAD INPUT FILES
# ─────────────────────────────────────────────────────────────

intermediate_dir = f"{OUTPUT_DIR}/intermediate_datasets"
decoys_path = f"{OUTPUT_DIR}/Generated_decoys.csv"

# Load datasets and automatically rename the SMILES column
df_actives_lude = rename_smiles_column(pd.read_csv(f"{intermediate_dir}/actives_for_lude_full_info.csv"))
df_actives_rem = rename_smiles_column(pd.read_csv(f"{intermediate_dir}/actives_remaining.csv"))
df_inactives_r = rename_smiles_column(pd.read_csv(f"{intermediate_dir}/inactives_real.csv"))

if os.path.exists(decoys_path):
    # Load and rename the decoy dataset
    df_decoy_raw = rename_smiles_column(pd.read_csv(decoys_path))
    # Ensure a standardized SMILES column is available
    df_decoy_raw = df_decoy_raw.dropna(subset=["smiles"])
else:
    print("❌ Error: decoy file not found!")
    df_decoy_raw = pd.DataFrame(columns=["smiles"])

# ─────────────────────────────────────────────────────────────
# 1. BUILD THE VIRTUAL SCREENING DATABASE
# (50 Actives + Decoys)
# ─────────────────────────────────────────────────────────────

print("\n--- BUILDING THE VIRTUAL SCREENING DATABASE ---")

df_vs_raw = pd.concat([
    df_actives_lude.assign(label=1),
    df_decoy_raw[["smiles"]].assign(label=0)
], ignore_index=True)

df_vs_final = process_database(df_vs_raw, "Virtual Screening Set")

# ─────────────────────────────────────────────────────────────
# 2. BUILD THE MAIN TRAINING DATABASE
# (Remaining Actives + Real Inactives)
# ─────────────────────────────────────────────────────────────

print("\n--- BUILDING THE MAIN TRAINING DATABASE ---")

df_main_raw = pd.concat([
    df_actives_rem.assign(label=1),
    df_inactives_r.assign(label=0)
], ignore_index=True)

df_main_final = process_database(df_main_raw, "Main Training Set")

# ─────────────────────────────────────────────────────────────
# 3. PREVENT DATA LEAKAGE AND EXPORT DATASETS
# ─────────────────────────────────────────────────────────────

# Remove compounds from the MAIN dataset that are also present
# in the VS dataset (e.g., identical canonical tautomers)
vs_smiles = set(df_vs_final["smiles"])
df_main_final = df_main_final[~df_main_final["smiles"].isin(vs_smiles)].copy()

# Output file paths
path_vs = f"{OUTPUT_DIR}/database_VS_final.csv"
path_main = f"{OUTPUT_DIR}/database_MAIN_final.csv"

df_vs_final.to_csv(path_vs, index=False)
df_main_final.to_csv(path_main, index=False)

print("\n" + "=" * 50)
print("🚀 PROCESS COMPLETED")
print(f"📍 VS Database: {path_vs} ({len(df_vs_final)} rows)")
print(f"📍 MAIN Database: {path_main} ({len(df_main_final)} rows)")
print("=" * 50)

In [ ]:
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────
# BLOCK 3 — DATASET STATISTICS AND VISUALIZATION
# ─────────────────────────────────────────────────────────────

def plot_and_stats(df, name, filename):
    print(f"\n📊 Class Distribution: {name}")
    print("-" * 30)
    print(df["label"].value_counts().rename({1: "Actives", 0: "Inactives/Decoys"}))

    # Keep only valid pChEMBL values (exclude NaN values from decoys)
    pchembl_values = df["pchembl"].dropna()

    if not pchembl_values.empty:
        plt.figure(figsize=(8, 6))
        plt.hist(pchembl_values, bins=40, color='skyblue', edgecolor='black', alpha=0.7)

        # Activity threshold
        plt.axvline(
            ACTIVE_THRESHOLD,
            color='red',
            linestyle='--',
            label=f'Threshold ({ACTIVE_THRESHOLD})'
        )

        plt.title(f"pChEMBL Distribution - {name}")
        plt.xlabel("pChEMBL value")
        plt.ylabel("Number of molecules")
        plt.grid(axis='y', alpha=0.3)
        plt.legend()

        # Save figure
        save_path = f"{OUTPUT_DIR}/{filename}.png"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"✅ Figure saved: {save_path}")
    else:
        print(f"ℹ️ No numeric pChEMBL values available for plotting in {name} (decoys only?)")

# --- Virtual Screening database ---
if not df_vs_final.empty:
    plot_and_stats(df_vs_final, "Virtual Screening Set (VS)", "dist_pchembl_VS")
else:
    print("⚠️ Error: VS database is empty!")

# --- Main Training database ---
if not df_main_final.empty:
    plot_and_stats(df_main_final, "Main Training Set", "dist_pchembl_MAIN")
    print("\n✅ Datasets ready")
else:
    print("⚠️ Error: MAIN database is empty!")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2. ROBUST FEATURIZATION (GENERATES BOTH VS AND MAIN OUTPUTS)
# ─────────────────────────────────────────────────────────────

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
import pandas as pd
import numpy as np
from tqdm import tqdm
import joblib
import os

# -------------------------------------------------------------
# ECFP4 FUNCTION IDENTICAL TO THE TRAINING PIPELINE
# -------------------------------------------------------------
def ecfp4_from_smiles(smiles):
    try:
        smiles = str(smiles).strip()
        mol = Chem.MolFromSmiles(smiles, sanitize=True)
        if mol is None:
            return None

        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=2,
            nBits=2048,
            useChirality=True
        )

        arr = np.zeros((2048,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr

    except Exception:
        return None

def process_and_featurize(file_path, prefix):
    """Load a dataset, generate fingerprints, and save the X and y matrices."""
    if not os.path.exists(file_path):
        print(f"⚠️ Skipping {prefix}: file not found at {file_path}")
        return None

    print(f"\n📦 Loading {prefix} dataset...")
    df = pd.read_csv(file_path)
    print(f"Compounds in CSV: {len(df)}")

    # Identify the SMILES column
    if "smiles" in df.columns:
        smiles_col = "smiles"
    elif "canonical_smiles" in df.columns:
        smiles_col = "canonical_smiles"
    else:
        raise ValueError(f"No SMILES column found in {prefix}")

    # Generate fingerprints
    fps = []
    valid_idx = []
    failed = 0

    print(f"🧬 Generating ECFP4 fingerprints for {prefix}...")
    for i, smi in enumerate(tqdm(df[smiles_col])):
        fp = ecfp4_from_smiles(smi)
        if fp is None:
            failed += 1
            continue
        fps.append(fp)
        valid_idx.append(i)

    print(f"\n---- FEATURIZATION REPORT ({prefix}) ----")
    print(f"Failed fingerprints: {failed}")
    print(f"Valid fingerprints : {len(fps)}")

    if len(fps) == 0:
        print(f"❌ Critical error: no valid fingerprints generated for {prefix}")
        return None

    # Synchronize the DataFrame and feature matrices
    df_valid = df.iloc[valid_idx].reset_index(drop=True)
    X = np.vstack(fps).astype(np.int8)
    y = df_valid["label"].values.astype(np.int8)

    # SAVE OUTPUTS
    joblib.dump(X, f"{OUTPUT_DIR}/X_{prefix}_ecfp4.joblib")
    joblib.dump(y, f"{OUTPUT_DIR}/y_{prefix}_labels.joblib")
    df_valid.to_csv(f"{OUTPUT_DIR}/dataset_{prefix}_featurized.csv", index=False)

    print(f"💾 {prefix} data saved (X shape: {X.shape})")
    return X, y

# ─────────────────────────────────────────────────────────────
# RUN FEATURIZATION ON BOTH DATASETS
# ─────────────────────────────────────────────────────────────

# 1. Process the MAIN database (training)
path_main = f"{OUTPUT_DIR}/database_MAIN_final.csv"
res_main = process_and_featurize(path_main, "MAIN")

# 2. Process the VS database (validation/decoys)
path_vs = f"{OUTPUT_DIR}/database_VS_final.csv"
res_vs = process_and_featurize(path_vs, "VS")

print("\n✅ Featurization completed for both datasets.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3. SCAFFOLD-BASED TRAIN/TEST SPLIT (80/20) + 1:1 CLASS BALANCING
# ─────────────────────────────────────────────────────────────

from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils import resample
import joblib
import numpy as np

# Reload the MAIN dataset generated in the previous block
# X and y refer to the features and labels of the MAIN database
X = joblib.load(f"{OUTPUT_DIR}/X_MAIN_ecfp4.joblib")
y = joblib.load(f"{OUTPUT_DIR}/y_MAIN_labels.joblib")
df = pd.read_csv(f"{OUTPUT_DIR}/dataset_MAIN_featurized.csv")

# -------------------------------------------------------------
# Murcko scaffold function
# -------------------------------------------------------------
def murcko_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    return "unknown"

print("\n🧱 Calculating Murcko scaffolds for the MAIN dataset...")
df["scaffold"] = df["smiles"].apply(murcko_scaffold)

# -------------------------------------------------------------
# 80/20 scaffold-based split
# -------------------------------------------------------------
scaffold_groups = df.groupby("scaffold")
scaffolds = list(scaffold_groups.groups.keys())

# Determine whether each scaffold is predominantly active or inactive
# to perform a stratified scaffold split
scaffold_labels = scaffold_groups["label"].mean()
scaffold_labels_bin = (scaffold_labels >= 0.5).astype(int).values

scaffold_idx = np.arange(len(scaffolds))

print("\n✂️ Performing scaffold-based 80/20 train/test split...")

sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=RANDOM_SEED
)

for train_sc_idx, test_sc_idx in sss.split(scaffold_idx, scaffold_labels_bin):
    train_scaffolds = [scaffolds[i] for i in train_sc_idx]
    test_scaffolds = [scaffolds[i] for i in test_sc_idx]

train_df = df[df["scaffold"].isin(train_scaffolds)].copy()
test_df = df[df["scaffold"].isin(test_scaffolds)].copy()

# Retrieve the feature matrices corresponding to the original indices
X_train_raw = X[train_df.index.values]
y_train_raw = y[train_df.index.values]
X_test = X[test_df.index.values]
y_test = y[test_df.index.values]

# ─────────────────────────────────────────────────────────────
# ⚖️ 1:1 CLASS BALANCING
# (Undersample the majority class in the training set)
# ─────────────────────────────────────────────────────────────
print("\n⚖️ Balancing the training set (target ratio: 1:1)...")

# Split training indices by class
idx_active = np.where(y_train_raw == 1)[0]
idx_inactive = np.where(y_train_raw == 0)[0]

n_min = min(len(idx_active), len(idx_inactive))

# Random undersampling
idx_active_resampled = resample(
    idx_active,
    n_samples=n_min,
    replace=False,
    random_state=RANDOM_SEED
)

idx_inactive_resampled = resample(
    idx_inactive,
    n_samples=n_min,
    replace=False,
    random_state=RANDOM_SEED
)

balanced_idx = np.concatenate([idx_active_resampled, idx_inactive_resampled])
np.random.shuffle(balanced_idx)

# Final balanced training matrices
X_train = X_train_raw[balanced_idx]
y_train = y_train_raw[balanced_idx]

# Update the training metadata to reflect the balanced dataset
train_df = train_df.iloc[balanced_idx].copy()

# -------------------------------------------------------------
# REMOVE SCAFFOLD COLUMN BEFORE SAVING
# -------------------------------------------------------------
train_df = train_df.drop(columns=["scaffold"])
test_df = test_df.drop(columns=["scaffold"])

# Load the VS dataset for final export
df_vs_clean = pd.read_csv(f"{OUTPUT_DIR}/dataset_VS_featurized.csv")
X_vs = joblib.load(f"{OUTPUT_DIR}/X_VS_ecfp4.joblib")
y_vs = joblib.load(f"{OUTPUT_DIR}/y_VS_labels.joblib")

# -------------------------------------------------------------
# SUMMARY REPORT
# -------------------------------------------------------------
print("\n📊 SPLIT AND BALANCING REPORT")
print(f"Training set (balanced 1:1): {len(train_df)} (Actives: {y_train.sum()})")
print(f"Test set (original):         {len(test_df)} (Actives: {y_test.sum()})")
print(f"VS set (screening):          {len(df_vs_clean)}")

# -------------------------------------------------------------
# SAVE QSAR DATASETS (MAIN SPLIT)
# -------------------------------------------------------------
joblib.dump(X_train, f"{OUTPUT_DIR}/X_train.joblib")
joblib.dump(X_test, f"{OUTPUT_DIR}/X_test.joblib")
joblib.dump(y_train, f"{OUTPUT_DIR}/y_train.joblib")
joblib.dump(y_test, f"{OUTPUT_DIR}/y_test.joblib")

train_df.to_csv(f"{OUTPUT_DIR}/train_metadata.csv", index=False)
test_df.to_csv(f"{OUTPUT_DIR}/test_metadata.csv", index=False)

# -------------------------------------------------------------
# SAVE VS DATASET (FINGERPRINTS + METADATA)
# -------------------------------------------------------------
joblib.dump(X_vs, f"{OUTPUT_DIR}/X_vs.joblib")
joblib.dump(y_vs, f"{OUTPUT_DIR}/y_vs.joblib")
df_vs_clean.to_csv(f"{OUTPUT_DIR}/vs_metadata.csv", index=False)

print(f"\n✅ Dataset split successfully saved to {OUTPUT_DIR}")
print("📦 Datasets are ready for Machine Learning and Virtual Screening")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4. CHEMICAL SPACE VISUALIZATION (PCA & t-SNE)
# ─────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

print("\n🎨 Generating chemical space visualizations...")

# Prepare data for visualization
# Combine the training and test sets while keeping track
# of their origin for plotting
X_combined = np.vstack([X_train, X_test])
set_labels = ["Train"] * len(X_train) + ["Test"] * len(X_test)
activity_labels = np.concatenate([y_train, y_test])

# 1. PCA (fast, captures the global variance)
pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_combined)

# 2. t-SNE (slower, but useful for visualizing
# local clusters and scaffold relationships)
# Use init='pca' for improved stability and
# perplexity=30 (standard value)
tsne = TSNE(
    n_components=2,
    perplexity=30,
    init='pca',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
X_tsne = tsne.fit_transform(X_combined)

# Create a DataFrame for Seaborn plotting
df_plot = pd.DataFrame({
    "PCA-1": X_pca[:, 0],
    "PCA-2": X_pca[:, 1],
    "tSNE-1": X_tsne[:, 0],
    "tSNE-2": X_tsne[:, 1],
    "Dataset": set_labels,
    "Activity": ["Active" if a == 1 else "Inactive" for a in activity_labels]
})

# --- Plotting ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: PCA colored by dataset (Train vs Test)
sns.scatterplot(
    data=df_plot,
    x="PCA-1",
    y="PCA-2",
    hue="Dataset",
    alpha=0.5,
    ax=axes[0],
    palette="Set1"
)
axes[0].set_title(
    f"PCA: Train vs Test Coverage\n"
    f"(Explained variance: {sum(pca.explained_variance_ratio_) * 100:.1f}%)"
)

# Plot 2: t-SNE colored by activity
sns.scatterplot(
    data=df_plot,
    x="tSNE-1",
    y="tSNE-2",
    hue="Activity",
    style="Dataset",
    alpha=0.6,
    ax=axes[1],
    palette={"Active": "red", "Inactive": "blue"}
)
axes[1].set_title("t-SNE: Chemical Clusters and Activity")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chemical_space_distribution.png", dpi=300)
plt.show()

print(f"✅ Figures saved to {OUTPUT_DIR}")

## MODEL TRAINING AND METRICS CALCULATION

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, matthews_corrcoef, 
    brier_score_loss, precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV

def enrichment_factor(y_true, y_score, k=0.01):
    n = max(1, int(len(y_true) * k))
    idx = np.argsort(y_score)[::-1][:n]
    # Handle cases with no positives to avoid division by zero
    total_pos = y_true.sum()
    if total_pos == 0: return 0.0
    return (y_true[idx].sum() / n) / (total_pos / len(y_true))

def compute_metrics(y_true, y_score):
    # Metrics
    fpr, tpr, _ = roc_curve(y_true, y_score)
    prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_score)
    
    results = {
        "ROC-AUC": auc(fpr, tpr),
        "PR-AUC":  auc(rec_curve, prec_curve),
        "Brier":   brier_score_loss(y_true, y_score),
        "EF1%":    enrichment_factor(y_true, y_score, 0.01),
        "EF5%":    enrichment_factor(y_true, y_score, 0.05),
    }

    # Metrics for multiple tresholds (0.5 e 0.7)
    for threshold in [0.5, 0.7]:
        y_pred = (y_score >= threshold).astype(int)
        suffix = f"@{int(threshold*100)}%"
        
        results[f"Precision{suffix}"] = precision_score(y_true, y_pred, zero_division=0)
        results[f"Recall{suffix}"]    = recall_score(y_true, y_pred, zero_division=0)
        results[f"F1{suffix}"]        = f1_score(y_true, y_pred, zero_division=0)
        results[f"MCC{suffix}"]       = matthews_corrcoef(y_true, y_pred)
        
    return results

# Initialization
base_rf = RandomForestClassifier(
    n_estimators=500, max_features="sqrt", class_weight="balanced",
    random_state=RANDOM_SEED, n_jobs=-1
)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
# Dynamic keys based on compute_metrics output
metric_keys = list(compute_metrics(np.array([0,1]), np.array([0.1, 0.9])).keys())
cv_results = {k: [] for k in metric_keys}

print("Starting Cross-Validation...")
for fold, (tr, te) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_te = X_train[tr], X_train[te]
    y_tr, y_te = y_train[tr], y_train[te]
 
    model = CalibratedClassifierCV(base_rf, method="sigmoid", cv=3)
    model.fit(X_tr, y_tr)
 
    y_score = model.predict_proba(X_te)[:, 1]
    m = compute_metrics(y_te, y_score)
 
    for k in metric_keys:
        cv_results[k].append(m[k])
    
    print(f"Fold {fold} | ROC-AUC: {m['ROC-AUC']:.3f} | MCC@50%: {m['MCC@50%']:.3f} | MCC@70%: {m['MCC@70%']:.3f}")

# --- CV SUMMARY CALCULATION ---
summary_data = []
for k in metric_keys:
    mean_val = np.mean(cv_results[k])
    std_val = np.std(cv_results[k])
    summary_data.append({"Metric": k, "Mean": mean_val, "Std": std_val})

summary_df = pd.DataFrame(summary_data).set_index("Metric")
summary_df.to_csv(f"{OUTPUT_DIR}/cv_metrics_detailed.csv")



## HOLDOUT TEST AND FINAL METRICS

In [ ]:
# --- FINAL TRAINING AND HOLDOUT ---
final_model = CalibratedClassifierCV(base_rf, method="sigmoid", cv=5)
final_model.fit(X_train, y_train)
joblib.dump(final_model, f"{MODEL_DIR}/qsar_ecfp4_model.joblib")

# Evaluation on Test Set
y_test_score = final_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_score >= 0.5).astype(int)
test_metrics = compute_metrics(y_test, y_test_score)


# --- BOOTSTRAP ANALYSIS (Holdout Set) ---
n_iterations = 100
bootstrap_stats = {k: [] for k in metric_keys}

print(f"Starting Bootstrap ({n_iterations} iterations)...")

for i in range(n_iterations):
    # Resample indices with replacement
    indices = np.arange(len(y_test))
    resample_idx = resample(indices, replace=True, random_state=RANDOM_SEED + i)
    
    # Calculate metrics for this bootstrap sample
    b_metrics = compute_metrics(y_test[resample_idx], y_test_score[resample_idx])
    
    for k in metric_keys:
        bootstrap_stats[k].append(b_metrics[k])

# Calculate Confidence Intervals (95%)
bootstrap_report = []
for k in metric_keys:
    values = np.sort(bootstrap_stats[k])
    lower = np.percentile(values, 2.5)
    upper = np.percentile(values, 97.5)
    mean_b = np.mean(values)
    bootstrap_report.append({
        "Metric": k,
        "Test_Score": test_metrics[k],
        "Boot_Mean": mean_b,
        "CI_Lower": lower,
        "CI_Upper": upper
    })

boot_df = pd.DataFrame(bootstrap_report).set_index("Metric")



# --- UPDATED SAVE TO TXT ---
with open(f"{OUTPUT_DIR}/final_report.txt", "w") as f:
    f.write("=== CROSS-VALIDATION SUMMARY (Mean ± Std) ===\n")
    f.write(summary_df.to_string())
    f.write("\n\n" + "="*40 + "\n")
    f.write("=== HOLDOUT TEST METRICS WITH 95% CI (Bootstrap) ===\n")
    f.write(boot_df.to_string())
    f.write("\n" + "="*40 + "\n")

# Also save the bootstrap details to CSV for potential plotting
boot_df.to_csv(f"{OUTPUT_DIR}/test_bootstrap_results.csv")

print(f"Bootstrap analysis complete. Detailed metrics saved to final_report.txt")


 
 

## SAVE PREDICTION

In [ ]:
# --- SAVE PREDICTIONS CSV ---
# Assuming 'test_metadata' contains 'smiles' and 'pChembl' for the X_test indices
predictions_df = pd.DataFrame({
    'SMILES': test_df['smiles'],
    'pChembl_Actual': test_df['pchembl'],
    'Label_Actual': y_test,
    'Label_Predicted': y_test_pred,
    'Probability': y_test_score
})
predictions_df.to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

print(f"\nProcess complete.")
print(f"Metrics saved to: {OUTPUT_DIR}/final_report.txt")
print(f"Predictions saved to: {OUTPUT_DIR}/test_predictions.csv")
 

## PLOT

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

# 1. ROC CURVE (Holdout Test)
plt.figure(figsize=(8, 6))
fpr, tpr, _ = roc_curve(y_test, y_test_score)
roc_auc_val = auc(fpr, tpr)

plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc_val:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'Receiver Operating Characteristic - Test\nAUC: {roc_auc_val:.3f}')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.savefig(f"{OUTPUT_DIR}/roc_curve_test.png", dpi=300, bbox_inches='tight')
plt.close()

# 2. PRECISION-RECALL CURVE (Holdout Test)
plt.figure(figsize=(8, 6))
prec, rec, _ = precision_recall_curve(y_test, y_test_score)
pr_auc_val = auc(rec, prec)

plt.plot(rec, prec, color='blue', lw=2, label=f'PR curve (area = {pr_auc_val:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'Precision-Recall Curve - Test\nPR-AUC: {pr_auc_val:.3f}')
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.savefig(f"{OUTPUT_DIR}/pr_curve_test.png", dpi=300, bbox_inches='tight')
plt.close()

# 3. CONFUSION MATRIX 

y_test_pred_50 = (y_test_score >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_test_pred_50)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Inactive', 'Active'], 
            yticklabels=['Inactive', 'Active'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix\nPrecision: {test_metrics["Precision@50%"]:.3f} | Recall: {test_metrics["Recall@50%"]:.3f}')
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix_test_50.png", dpi=300, bbox_inches='tight')
plt.close()

# 4. SAVING TEST RESULTS
pd.DataFrame({
    "true_label": y_test, 
    "probability": y_test_score, 
    "pred": y_test_pred_50
}).to_csv(f"{OUTPUT_DIR}/final_test_predictions_scores.csv", index=False)

print("Graph saved with success!")

In [ ]:
# Feature importance analysis

rf_full      = base_rf.fit(X_train, y_train)   
importances  = rf_full.feature_importances_
top_idx      = np.argsort(importances)[::-1][:20]
 
plt.figure()
plt.bar(range(20), importances[top_idx])
plt.xticks(range(20), top_idx, rotation=90)
plt.title("Top 20 ECFP4 bits (RF feature importance)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/feature_importance.png", dpi=300)
plt.show()
plt.close()

## SHAP ANALYSIS

In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import joblib


rf_full_loaded = joblib.load(f"{MODEL_DIR}/qsar_ecfp4_model.joblib")
X_test_loaded = joblib.load(f"{OUTPUT_DIR}/X_test.joblib")


if hasattr(rf_full_loaded, "calibrated_classifiers_"):
    model_to_explain = rf_full_loaded.calibrated_classifiers_[0].estimator
else:
    model_to_explain = rf_full_loaded

print("🚀 Inizialization Explainer...")

try:
    explainer = shap.Explainer(model_to_explain, X_test_loaded)
    
   
    X_sub = X_test_loaded[:400]
    shap_values = explainer(X_sub, check_additivity=False)

    
    if len(shap_values.shape) == 3:
        shap_values_pos = shap_values[:, :, 1]
    else:
        shap_values_pos = shap_values

    # 4. PLOTTING - BEESWARM
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values_pos, X_sub, max_display=20, plot_type="dot", show=False)
    plt.title("Top 20 ECFP4 bits — SHAP Beeswarm")
    plt.savefig(f"{OUTPUT_DIR}/SHAP_beeswarm.png", dpi=300, bbox_inches="tight")
    plt.close()

    # ---  PLOTTING - BAR PLOT ---
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values_pos, X_sub, max_display=20, plot_type="bar", show=False)
    plt.title("Global Feature Importance — SHAP Bar")
    plt.savefig(f"{OUTPUT_DIR}/SHAP_bar.png", dpi=300, bbox_inches="tight")
    plt.close()
    # -------------------------------------

    
    mean_abs_shap = np.abs(shap_values_pos.values).mean(axis=0)
    mean_shap     = shap_values_pos.values.mean(axis=0)
    top_indices   = np.argsort(mean_abs_shap)[::-1][:40]

    positive_bits = []
    negative_bits = []

    for idx in top_indices:
        bit_id = int(idx)
        if mean_shap[idx] > 0 and len(positive_bits) < 6:
            positive_bits.append(bit_id)
        elif mean_shap[idx] < 0 and len(negative_bits) < 6:
            negative_bits.append(bit_id)

    print("✅  SHAP completed!")
    print(f"📊 Grafici salvati in: {OUTPUT_DIR}")
    print("✅ Positive Bit :", positive_bits)
    print("✅ Negative Bit :", negative_bits)

except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"❌ Error during SHAP: {e}")

In [ ]:
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D

def draw_fragment(mol, atoms, bonds, path, positive=True):
    """
    Generate bit images
    """
    # Color: green for positive impact, red for negative impact
    color = (0.2, 0.9, 0.2) if positive else (0.9, 0.2, 0.2)
    
    
    atom_colors = {a: color for a in atoms}
    bond_colors = {b: color for b in bonds}
    
    
    drawer = rdMolDraw2D.MolDraw2DCairo(300, 300)
    options = drawer.drawOptions()
    options.useBWAtomPalette() 
    
    
    drawer.DrawMolecule(
        mol, 
        highlightAtoms=atoms, 
        highlightAtomColors=atom_colors,
        highlightBonds=bonds,
        highlightBondColors=bond_colors
    )
    drawer.FinishDrawing()
    
    # Save
    with open(path, 'wb') as f:
        f.write(drawer.GetDrawingText())

In [ ]:
import os
import joblib
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm


FRAG_DIR = os.path.join(OUTPUT_DIR, "fragments")
os.makedirs(FRAG_DIR, exist_ok=True)

BITS_TO_MAP = {
    "Positive": positive_bits,
    "Negative": negative_bits
}


mapped_bits = {bit: False for group in BITS_TO_MAP.values() for bit in group}


test_metadata = pd.read_csv(f"{OUTPUT_DIR}/test_metadata.csv")

print(f"Scansione di {len(test_metadata)} molecole per trovare i bit significativi...")


for i, smi in enumerate(tqdm(test_metadata['smiles'], desc="Mapping fragments")):
    mol = Chem.MolFromSmiles(smi)
    if not mol: 
        continue
    
    bit_info = {}
   
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol, radius=2, nBits=2048, useChirality=True, bitInfo=bit_info
    )
    
    for effect, bits in BITS_TO_MAP.items():
        for bit in bits:
            
            if mapped_bits[bit] or bit not in bit_info:
                continue
            
            
            atom_id, radius = bit_info[bit][0]
            
            if radius > 0:
                env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_id)
                atoms = set()
                for b_idx in env:
                    atoms.add(mol.GetBondWithIdx(b_idx).GetBeginAtomIdx())
                    atoms.add(mol.GetBondWithIdx(b_idx).GetEndAtomIdx())
                bonds = list(env)
            else:
                atoms = [atom_id]
                bonds = []
            
            filename = f"fragment_{bit}_{effect}.png"
            path = os.path.join(FRAG_DIR, filename)
            
           
            try:
                draw_fragment(mol, list(atoms), bonds, path, positive=(effect=="Positive"))
                mapped_bits[bit] = True
                print(f"📸 Salvata immagine per bit: {bit} ({effect})")
            except NameError:
                print("❌ Errore: la funzione 'draw_fragment' non è definita nel tuo script.")
                break

    
    if all(mapped_bits.values()): 
        break

print(f"\nFine! Immagini generate: {sum(mapped_bits.values())} su {len(positive_bits) + len(negative_bits)}")

In [ ]:
from IPython.display import Image, display
path = f"{FRAG_DIR}/fragment_{bit}_{effect}.png"
display(Image(path))

## AD CALCULATION

In [ ]:
from sklearn.decomposition import PCA
from scipy.spatial.distance import mahalanobis
import joblib
import numpy as np
import pandas as pd

# --- PCA RECONSTRUCTION ---
print("⚙️ Initializing PCA from the Training Set...")
X_train = joblib.load(f"{OUTPUT_DIR}/X_train.joblib")

# Use 50 components (a good compromise between compression and information retention)
pca = PCA(n_components=50, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train)
joblib.dump(pca, f"{OUTPUT_DIR}/pca_transformer.joblib")

# --- MAHALANOBIS PARAMETERS ---
mu = np.mean(X_train_pca, axis=0)
V = np.cov(X_train_pca, rowvar=False)
IV = np.linalg.inv(V)

# Compute Mahalanobis distances for the training set
train_distances = [mahalanobis(x, mu, IV) for x in X_train_pca]
threshold_ad = np.percentile(train_distances, 95)

print(f"✅ PCA ready. Mahalanobis threshold (95th percentile): {threshold_ad:.4f}")

# --- APPLY TO THE TEST SET ---
print("\n🧪 Evaluating the Applicability Domain on the Test Set...")
X_test = joblib.load(f"{OUTPUT_DIR}/X_test.joblib")
X_test_pca = pca.transform(X_test)

test_distances = [mahalanobis(x, mu, IV) for x in X_test_pca]
inside_ad = [d <= threshold_ad for d in test_distances]

perc_inside = (sum(inside_ad) / len(inside_ad)) * 100
print(f"📊 Result: {perc_inside:.1f}% of the Test Set molecules are within the Applicability Domain")

# Optional: Save the results for future analyses
test_metadata = pd.read_csv(f"{OUTPUT_DIR}/test_metadata.csv")
test_metadata['mahalanobis_dist'] = test_distances
test_metadata['in_ad'] = inside_ad
test_metadata.to_csv(f"{OUTPUT_DIR}/test_results_with_ad.csv", index=False)

## PREDIZIONE SU ATTIVI E DECOY

In [ ]:
def predict_vs(df_vs, X_vs):

    print("\n🌍 Evaluating the Applicability Domain...")

    X_pca = pca.transform(X_vs)
    distances = np.array([mahalanobis(x, mu, IV) for x in X_pca])

    df_vs["mahalanobis_dist"] = distances
    df_vs["ad_status"] = np.where(distances <= threshold_ad, "Inside", "Outside")

    print("\n🤖 Loading the QSAR model...")
    model = joblib.load(f"{MODEL_DIR}/qsar_ecfp4_model.joblib")

    print("🔮 Predicting probabilities...")
    probs = model.predict_proba(X_vs)[:, 1]
    preds = model.predict(X_vs)

    df_vs["probability"] = probs
    df_vs["pred_label"] = preds

    return df_vs

In [ ]:
def enrichment_factor(y_true, y_score, top=0.01):
    n_total = len(y_true)
    n_top = int(n_total * top)
    
    idx = np.argsort(y_score)[::-1][:n_top]
    hits_top = y_true[idx].sum()
    hits_total = y_true.sum()
    
    ef = (hits_top / n_top) / (hits_total / n_total)
    return ef

In [ ]:
def dude_benchmark_vs(df_vs, X_vs):

    print("\n==============================")
    print("🚀 AVVIO VS BENCHMARK QSAR")
    print("==============================")

    # -----------------------------
    # PREDICTION
    # -----------------------------
    df_vs = predict_vs(df_vs, X_vs)

    # -----------------------------
    # METRICs VS
    # -----------------------------
    print("\n📈 Calculating virtual screening metrics...")

    y_true = df_vs["label"].values  
    y_score = df_vs["probability"].values

    auc = roc_auc_score(y_true, y_score)
    ef1 = enrichment_factor(y_true, y_score, 0.01)
    ef5 = enrichment_factor(y_true, y_score, 0.05)
    ef10 = enrichment_factor(y_true, y_score, 0.10)

    print(f"\nAUC ROC : {auc:.3f}")
    print(f"EF 1%   : {ef1:.2f}")
    print(f"EF 5%   : {ef5:.2f}")
    print(f"EF 10%  : {ef10:.2f}")

    # -----------------------------
    # SAVE OUTPUT
    # -----------------------------
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    out_csv = os.path.join(OUTPUT_DIR, "VS_QSAR_predictions.csv")
    df_vs.to_csv(out_csv, index=False)

    print("\n💾 Saved:", out_csv)

In [ ]:

df_vs = pd.read_csv(f"{OUTPUT_DIR}/vs_metadata.csv")


X_vs = np.vstack([
    ecfp4_from_smiles(smi)
    for smi in df_vs["smiles"]
]).astype(np.int8)

print("X_vs shape:", X_vs.shape)

dude_benchmark_vs(df_vs, X_vs)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_true = df_vs["label"].values  
y_score = df_vs["probability"].values

fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
plt.savefig(f"{OUTPUT_DIR}/roc_curve_vs.png", dpi=300, bbox_inches='tight')

from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, _ = precision_recall_curve(y_true, y_score)
ap = average_precision_score(y_true, y_score)

plt.figure()
plt.plot(recall, precision, label=f"AP = {ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
plt.savefig(f"{OUTPUT_DIR}/roc_curve_vs.png", dpi=300, bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df_vs[df_vs["label"]==1]["probability"], bins=30, alpha=0.6, label="Actives")
plt.hist(df_vs[df_vs["label"]==0]["probability"], bins=30, alpha=0.6, label="Decoys")
plt.legend()
plt.title("Score distribution")
plt.show()

In [ ]:
plt.scatter(df_vs["mahalanobis_dist"], df_vs["probability"],
            c=df_vs["label"], cmap="coolwarm", alpha=0.6)
plt.axvline(threshold_ad, linestyle="--")
plt.xlabel("Mahalanobis distance")
plt.ylabel("Predicted probability")
plt.title("Applicability Domain vs Prediction")
plt.show()

In [ ]:
import numpy as np

y_true = df_vs["label"].values
probs = df_vs["probability"].values

N_total = len(df_vs)
N_actives = np.sum(y_true)

print("DATASET ORIGINALE")
print("Total molecules:", N_total)
print("Real actives:", N_actives)
print("Actives rate: %.3f" % (N_actives / N_total))

In [ ]:
threshold = 0.5
pred_active = probs >= threshold

N_pass = np.sum(pred_active)
N_actives_recovered = np.sum(y_true[pred_active])

print("\nQSAR FILTER (optimized threshold)")
print("Molecules passing the filter:", N_pass)
print("Dataset reduction: %.2f%%" % (100 * (1 - N_pass / N_total)))
print("Recovered actives:", N_actives_recovered)
print("Active recall: %.2f%%" % (100 * N_actives_recovered / N_actives))

In [ ]:
high_conf = probs >= 0.7

N_high = np.sum(high_conf)
N_high_actives = np.sum(y_true[high_conf])

print("\nHIGH-CONFIDENCE PREDICTIONS (P > 0.7)")
print("Molecules:", N_high)
print("Percentage of the dataset: %.2f%%" % (100 * N_high / N_total))
print("True actives among these:", N_high_actives)

if N_high > 0:
    print("High-confidence precision: %.2f%%" % (100 * N_high_actives / N_high))

In [ ]:
print("\nOPERATIONAL SUMMARY")
print("-----------------------")
print("Initial dataset:", N_total)
print("Docking subset:", N_pass)
print("High-confidence subset:", N_high)